**Navigation** : [Index](../../README.md) | [<< Précédent](01-1-OpenAI-DALL-E-3.ipynb) | [Suivant >>](01-3-Basic-Image-Operations.ipynb)

# Génération et analyse d'images par API multimodale

**Module :** 01-Images-Foundation  
**Niveau :** 🟢 Débutant  
**Technologies :** GPT-5, OpenRouter API, Vision AI  
**Durée estimée :** 30 minutes  

## 🎯 Objectifs d'Apprentissage

- [ ] Configurer GPT-5 via OpenRouter pour analyse d'images
- [ ] Maîtriser les conversations multimodales texte + image
- [ ] Analyser et décrire des images avec précision
- [ ] Créer des prompts optimisés pour l'analyse visuelle
- [ ] Intégrer GPT-5 dans des cas d'usage pédagogiques

## 📚 Prérequis

- Environment Setup (module 00) complété
- Clé API OpenRouter configurée
- Connaissances de base en IA multimodale

## ⚡ Capacités GPT-5

- **Vision avancée** : Analyse détaillée d'images
- **Multimodal** : Conversation texte + image simultanée
- **Contexte étendu** : Jusqu'à 400K tokens [OpenAI 2025]
- **Raisonnement** : Analyse complexe et déductive
- **Éducatif** : Parfait pour l'enseignement

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration conversation
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"               

# Parametres GPT-5
model_name = "openai/gpt-5"        # Modele via OpenRouter
max_tokens = 4000                  # Tokens de reponse max
temperature = 0.7                  # Creativite (0.0-1.0)
top_p = 0.9                        # Diversite sampling

# Configuration analyse
analyze_images = True              # Analyser les images (False pour validation structurelle seule)
analysis_mode = "detailed"         # "quick", "detailed", "educational"
include_technical_details = True   # Details techniques images
export_analysis = True             # Sauvegarder analyses
generate_alt_text = True           # Generer descriptions accessibilite

# Parametres pedagogiques
educational_level = "university"   # "elementary", "secondary", "university"
language = "francais"              # Langue des explications
include_examples = True            # Inclure exemples pratiques

In [2]:
# Parameters
BATCH_MODE = "true"


## Setup de l'environnement

Chargeons les dependances, configurons l'API OpenRouter pour acceder a GPT-5, et preparons les fonctions utilitaires pour le traitement d'images.

In [3]:
# Verification des dependances externes
import importlib

_DEPS_STATUS = {}
try:
    importlib.import_module('PIL')
    _DEPS_STATUS['PIL'] = True
except ImportError:
    _DEPS_STATUS['PIL'] = False
    print(f'WARNING: Pillow non installe - pip install Pillow')

try:
    importlib.import_module('requests')
    _DEPS_STATUS['requests'] = True
except ImportError:
    _DEPS_STATUS['requests'] = False
    print(f'WARNING: requests non installe - pip install requests')

try:
    importlib.import_module('matplotlib')
    _DEPS_STATUS['matplotlib'] = True
except ImportError:
    _DEPS_STATUS['matplotlib'] = False
    print(f'WARNING: matplotlib non installe - pip install matplotlib')

try:
    importlib.import_module('dotenv')
    _DEPS_STATUS['dotenv'] = True
except ImportError:
    _DEPS_STATUS['dotenv'] = False
    print(f'WARNING: python-dotenv non installe - pip install python-dotenv')

_all_deps_ok = all(_DEPS_STATUS.values())
if not _all_deps_ok:
    missing = [k for k, v in _DEPS_STATUS.items() if not v]
    print(f'Dependances manquantes: {missing}')
else:
    print('Toutes les dependances sont disponibles')

# Setup environnement et imports
import os
import sys
import json
import requests
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Union
import base64
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import logging
from urllib.parse import urlparse

# Chargement robuste de la configuration .env
from dotenv import load_dotenv

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# GENAI_ROOT pointe vers le dossier GenAI
HELPERS_PATH = current_path / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.genai_helpers import setup_genai_logging, load_genai_config
        print("Helpers GenAI importes")
    except ImportError:
        print("Helpers GenAI non disponibles - mode autonome")

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('gpt5_multimodal')

print(f"GPT-5 Multimodal - Analyse et Generation d'Images")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {notebook_mode}, Analyse: {analysis_mode}, Niveau: {educational_level}")
print(f"Langue: {language}, Max tokens: {max_tokens}")

Toutes les dependances sont disponibles


.env charge depuis: .env
Helpers GenAI importes
GPT-5 Multimodal - Analyse et Generation d'Images
2026-11-05 00:20:34
Mode: interactive, Analyse: detailed, Niveau: university
Langue: francais, Max tokens: 4000


L'environnement est initialise. La cellule suivante configure l'acces a l'API OpenRouter et verifie que le modèle GPT-5 est disponible pour les requêtes multimodales.


### Lecture du résultat — initialisation de l'environnement

La cellule précédente atteste que les quatre dépendances critiques sont résolues sur la machine hôte : **Pillow** (encodage base64 des images), **requests** (POST `/chat/completions` vers OpenRouter), **matplotlib** (visualisation locale optionnelle des images) et **python-dotenv** (chargement du fichier `.env`). Le message `Toutes les dependances sont disponibles` est la condition préalable au pipeline multimodal — sans elle, l'appel à GPT-5 lèverait une `ImportError` au moment de l'encodage base64, pas au moment de la requête HTTP.

Le bloc suivant charge la configuration utilisateur depuis `.env` (clé `OPENROUTER_API_KEY`, base URL, variables additionnelles). L'affichage final résume les paramètres du notebook : `Mode: interactive, Analyse: detailed, Niveau: university, Langue: francais, Max tokens: 4000`. Ces valeurs sont celles que les cellules suivantes vont passer en arguments à `analyze_image_with_gpt5()` — toute modification d'un paramètre dans la cellule 1 se propage ici sans ré-exécuter le reste du notebook.

In [4]:
# Configuration API OpenRouter pour GPT-5
print("\n🔑 CONFIGURATION GPT-5 MULTIMODAL")
print("=" * 42)

# Vérification clé API
openrouter_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('OPENAI_API_KEY')
if not openrouter_key:
    # Fallback pour validation structurelle sans clé
    if notebook_mode == "batch" and not analyze_images:
        print("⚠️  Mode batch sans analyse : Clé API ignorée")
        openrouter_key = "dummy_key_for_validation"
    else:
        raise ValueError(
            "OPENROUTER_API_KEY manquante dans .env\n"
            "Configurez votre clé dans le fichier .env (voir .env.example)"
        )

print(f"✅ Clé API OpenRouter configurée")

# Configuration headers et endpoint
api_base_url = os.getenv("OPENROUTER_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"))
headers = {
    "Authorization": f"Bearer {openrouter_key}",
    "HTTP-Referer": "https://coursia.myia.io",
    "X-Title": "CoursIA GenAI Images - GPT-5 Multimodal",
    "Content-Type": "application/json"
}

# Test connexion et vérification modèle GPT-5
if openrouter_key != "dummy_key_for_validation":
    try:
        response = requests.get(f"{api_base_url}/models", headers=headers, timeout=10)
        if response.status_code == 200:
            models_data = response.json()
            gpt5_models = [m for m in models_data.get('data', []) if 'gpt-5' in m.get('id', '').lower()]
            
            if gpt5_models:
                print(f"✅ Connexion réussie - {len(gpt5_models)} modèles GPT-5 disponibles")
                
                for model in gpt5_models:
                    print(f"  🧠 {model['id']} - Contexte: {model.get('context_length', 'N/A')} tokens")
                    if 'vision' in model.get('capabilities', []):
                        print(f"     👁️  Capacités vision activées")
            else:
                print(f"⚠️  Aucun modèle GPT-5 détecté - vérifiez votre accès")
                print(f"🔍 Modèles disponibles avec 'gpt' : {len([m for m in models_data.get('data', []) if 'gpt' in m.get('id', '').lower()])}")
        else:
            print(f"⚠️  Connexion API: HTTP {response.status_code}")
    except Exception as e:
        print(f"❌ Erreur connexion: {str(e)[:100]}...")
else:
    print("⏭️  Test connexion API sauté (dummy key)")
    
print(f"\n🎯 Modèle sélectionné: {model_name}")
print(f"⚙️  Paramètres: Temperature={temperature}, Max tokens={max_tokens}")
print(f"📊 Mode d'analyse: {analysis_mode}")



🔑 CONFIGURATION GPT-5 MULTIMODAL
✅ Clé API OpenRouter configurée


✅ Connexion réussie - 46 modèles GPT-5 disponibles
  🧠 openai/gpt-5.6-luna-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-luna:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-terra:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol - Contexte: 1050000 tokens
  🧠 openai/gpt-5.6-sol:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5-pro:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5 - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5:batch - Contexte: 1050000 tokens
  🧠 openai/gpt-5.4-image-2 - Contexte: 272000 tokens
  🧠 openai/gpt-5.4-n

L'API est configuree et la connexion validee. La cellule suivante définit les fonctions d'analyse multimodale qui encapsulent les appels a GPT-5 avec encodage des images en base64.


### Lecture du résultat — connexion à OpenRouter

La cellule précédente a effectué un `GET /models` sur le point d'entrée OpenRouter configuré. La réponse `HTTP/1.1 200 OK` confirme que la clé d'API est valide et que la passerelle reconnait le compte. L'extraction `gpt5_models` filtre les modèles dont l'identifiant contient `gpt-5` — le compte dépend du catalogue de la passerelle au moment de l'appel (la sortie ci-dessus affiche l'inventaire complet, de l'ordre de la cinquantaine à l'exécution de référence). C'est l'écart le plus visible entre la nomenclature officielle (OpenAI publie `gpt-5`, `gpt-5-mini`, `gpt-5-pro`, etc.) et la nomenclature passerelle (chaque fournisseur apparaît avec son préfixe : `openai/gpt-5`, `openai/gpt-5.5-pro`, `openai/gpt-5.4-image-2`, `openai/gpt-5.3-codex`…). Cette explosion taxonomique est l'effet de la stratégie OpenRouter d'exposer un identifiant par couple `(fournisseur, variante)`.

Chaque ligne `Contexte: N tokens` de la sortie donne la fenêtre du modèle, et c'est cette ligne qu'il faut lire modèle par modèle : la fenêtre ne suit aucune règle par suffixe. La sortie ci-dessus en donne l'étendue — de `gpt-5.2-chat` (128 000 tokens) aux générations `gpt-5.6`/`gpt-5.5`/`gpt-5.4` et leurs déclinaisons `-pro` (fenêtre million-tokens) — et dément les intuitions de famille : `gpt-5.2-pro` et `gpt-5-pro` restent à 400 000 quand `gpt-5.5-pro` monte à 1 050 000 ; côté image, `gpt-5-image` affiche 400 000 quand seule la variante `gpt-5.4-image-2` descend à 272 000. Un message image plus un long prompt d'analyse multimodal tiennent dans chacune de ces fenêtres, et les fenêtres million-tokens ouvrent la voie à des workflows multi-tours où l'analyse préliminaire est rappelée dans chaque question suivante.

In [5]:
# Fonctions utilitaires pour traitement d'images
def encode_image_base64(image_path: Union[str, Path]) -> str:
    """
    Encode une image locale en base64 pour GPT-5.
    
    Args:
        image_path: Chemin vers l'image locale
        
    Returns:
        String base64 de l'image
    """
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur lecture image {image_path}: {str(e)}")

def download_and_encode_image(image_url: str) -> str:
    """
    Télécharge et encode une image depuis URL.
    
    Args:
        image_url: URL de l'image
        
    Returns:
        String base64 de l'image
    """
    try:
        response = requests.get(image_url, timeout=30)
        response.raise_for_status()
        return base64.b64encode(response.content).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur téléchargement {image_url}: {str(e)}")

def prepare_image_for_gpt5(image_source: Union[str, Path]) -> Dict[str, str]:
    """
    Prépare une image pour GPT-5 (locale ou URL).
    
    Args:
        image_source: Chemin local ou URL de l'image
        
    Returns:
        Dict avec format attendu par GPT-5
    """
    if isinstance(image_source, (str, Path)):
        str_source = str(image_source)
        
        # Vérification URL
        if str_source.startswith(('http://', 'https://')):
            try:
                # Pour les URLs, GPT-5 peut les traiter directement
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": str_source
                    }
                }
            except:
                # Fallback : télécharger et encoder
                base64_image = download_and_encode_image(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
        else:
            # Image locale
            if Path(str_source).exists():
                base64_image = encode_image_base64(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            else:
                raise FileNotFoundError(f"Image non trouvée: {str_source}")
    
    raise ValueError(f"Format d'image non supporté: {type(image_source)}")

print("✅ Fonctions utilitaires images prêtes")

✅ Fonctions utilitaires images prêtes


### Fonction d'analyse et images de demonstration

La cellule suivante définit la fonction principale `analyze_image_with_gpt5()` qui encode une image en base64 et l'envoie a l'API multimodale. On prepare ensuite un ensemble d'images de test.


### Lecture du résultat — pipeline d'encodage des images

La cellule précédente expose trois fonctions utilitaires qui forment le pipeline d'encodage : `encode_image_base64()` (lecture locale + encodage base64), `download_and_encode_image()` (GET HTTP + encodage pour les URLs distantes) et `prepare_image_for_gpt5()` (dispatcher qui choisit entre URL directe et base64 selon le préfixe). Le retour uniforme est un dict `{type: image_url, image_url: {url: ...}}` qui correspond à la spécification OpenAI du champ `content` multimodal — c'est ce format que `analyze_image_with_gpt5()` passera ensuite dans le tableau `messages`.

Le `print('✅ Fonctions utilitaires images prêtes')` est un signal de chargement, pas un test d'exécution. Pour vérifier que le pipeline fonctionne réellement, il faut passer à la cellule d'analyse sur une image concrète — la fonction `prepare_image_for_gpt5` n'est validée qu'à ce moment-là, et un échec (URL inaccessible, fichier introuvable, format non supporté) remonte une `ValueError` ou `FileNotFoundError` typée.

In [6]:
# Fonction principale d'analyse d'image avec GPT-5
def analyze_image_with_gpt5(image_source: Union[str, Path], 
                           prompt: str = None,
                           analysis_type: str = "detailed") -> Dict[str, Any]:
    """
    Analyse une image avec GPT-5 multimodal.
    
    Args:
        image_source: Chemin local ou URL de l'image
        prompt: Prompt personnalisé (optionnel)
        analysis_type: Type d'analyse ("quick", "detailed", "educational")
        
    Returns:
        Dict avec analyse complète
    """
    
    # Prompts prédéfinis selon le type d'analyse
    analysis_prompts = {
        "quick": f"Décris brièvement cette image en {language}. Sois concis mais précis.",
        
        "detailed": f"""Analyse cette image en détail en {language}. Inclus :
        
1. **Description générale** : Que voit-on dans l'image ?
2. **Éléments visuels** : Couleurs, composition, style artistique
3. **Contexte** : Époque, lieu, situation probable
4. **Détails techniques** : Qualité, résolution apparente, type de photo/illustration
5. **Émotions/Atmosphère** : Quelle ambiance dégage l'image ?
6. **Interprétation** : Signification possible, message artistique

Sois précis et pédagogique dans tes explications.""",
        
        "educational": f"""Tu es un professeur expert analysant cette image pour des étudiants de niveau {educational_level}. En {language}, fournis :

🎯 **ANALYSE PÉDAGOGIQUE**

**1. Description accessible**
- Que montre cette image de façon simple et claire ?

**2. Éléments à observer**
- Quels détails importants les étudiants doivent-ils remarquer ?
- Techniques artistiques ou photographiques utilisées

**3. Contexte éducatif**
- Dans quel domaine d'étude cette image serait-elle utile ?
- Quelles disciplines académiques peuvent l'utiliser ?

**4. Questions pour réflexion**
- 3 questions que tu poserais aux étudiants sur cette image

**5. Connexions interdisciplinaires**
- Comment cette image se connecte-t-elle à d'autres sujets ?

Adapte ton vocabulaire au niveau {educational_level}."""
    }
    
    # Sélection du prompt
    if prompt is None:
        prompt = analysis_prompts.get(analysis_type, analysis_prompts["detailed"])
    
    try:
        print(f"\n🔍 Analyse en cours...")
        print(f"📝 Type: {analysis_type}")
        print(f"🖼️  Source: {str(image_source)[:100]}{'...' if len(str(image_source)) > 100 else ''}")
        
        # Préparation de l'image
        image_data = prepare_image_for_gpt5(image_source)
        
        # Construction du message multimodal
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    image_data
                ]
            }
        ]
        
        # Payload de la requête
        payload = {
            "model": model_name,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p
        }
        
        # Requête API
        start_time = datetime.now()
        response = requests.post(
            f"{api_base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        response_time = (datetime.now() - start_time).total_seconds()
        
        if response.status_code == 200:
            result = response.json()
            
            analysis_text = result["choices"][0]["message"]["content"]
            
            # Métadonnées de l'analyse
            metadata = {
                "model": model_name,
                "analysis_type": analysis_type,
                "educational_level": educational_level,
                "language": language,
                "timestamp": datetime.now().isoformat(),
                "response_time": response_time,
                "tokens_used": result.get("usage", {}),
                "image_source": str(image_source),
                "prompt_length": len(prompt)
            }
            
            return {
                "success": True,
                "analysis": analysis_text,
                "metadata": metadata,
                "image_source": image_source,
                "analysis_type": analysis_type
            }
        else:
            error_data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
            error_msg = error_data.get("error", {}).get("message", f"HTTP {response.status_code}")
            
            return {
                "success": False,
                "error": error_msg,
                "image_source": image_source,
                "status_code": response.status_code
            }
            
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "image_source": image_source
        }

print("✅ Fonction d'analyse GPT-5 prête")

✅ Fonction d'analyse GPT-5 prête


Les exemples d'images et les prompts pedagogiques sont définis. La cellule suivante lance l'analyse de demonstration en testant les trois modes (quick, detailed, educational) sur une image concrete.


### Lecture du résultat — fonction `analyze_image_with_gpt5`

La cellule précédente assemble trois prompts prédéfinis (`quick`, `detailed`, `educational`) sélectionnés par l'argument `analysis_type`. Le prompt `educational` est le plus structurant : il impose le rôle, le niveau éducatif (lu depuis `educational_level`), la langue (`language`), et un format de sortie à cinq sections explicites (`Description accessible`, `Éléments à observer`, `Contexte éducatif`, `Questions pour réflexion`, `Connexions interdisciplinaires`). Cette rigidité du prompt est ce qui garantit la reproductibilité des analyses entre deux sessions.

Le bloc d'appel HTTP mesure le temps de réponse (`response_time`) et récupère `usage` qui contient `prompt_tokens`, `completion_tokens` et `total_tokens`. Le champ `reasoning_tokens` (présent dans la sortie de GPT-5 image-generation) reflète les tokens consommés en phase de raisonnement interne — c'est un coût invisible dans `completion_tokens` seul, mais facturé. La cellule suivante utilise cette instrumentation pour comparer les trois modes sur une même image.

In [7]:
# Exemples d'images pour démonstration
print("\n🖼️  EXEMPLES D'ANALYSE GPT-5")
print("=" * 40)

# Images d'exemple (URLs publiques pour tests)
example_images = [
    {
        "title": "🏛️ Architecture Historique",
        "url": "https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800",  # Colisée Rome
        "description": "Monument historique romain - idéal pour analyse architecturale",
        "category": "Histoire/Architecture"
    },
    {
        "title": "🔬 Science et Technologie",
        "url": "https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800",  # Laboratoire
        "description": "Environnement scientifique - parfait pour analyse technique",
        "category": "Science/Technologie"
    },
    {
        "title": "🎨 Art et Culture",
        "url": "https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800",  # Peinture
        "description": "Œuvre artistique - excellent pour analyse esthétique",
        "category": "Art/Culture"
    },
    {
        "title": "🌍 Nature et Environnement",
        "url": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800",  # Paysage naturel
        "description": "Paysage naturel - parfait pour analyse géographique",
        "category": "Géographie/Environnement"
    }
]

# Affichage des exemples
for i, example in enumerate(example_images, 1):
    print(f"\n{i}. {example['title']}")
    print(f"   📂 Catégorie: {example['category']}")
    print(f"   📝 {example['description']}")
    print(f"   🔗 URL: {example['url'][:60]}...")

print(f"\n💡 Conseils pour l'analyse avec GPT-5:")
print(f"• Utilisez des images de haute qualité")
print(f"• Posez des questions spécifiques")
print(f"• Exploitez le contexte éducatif")
print(f"• Combinez analyse textuelle et visuelle")
print(f"• Adaptez le niveau de complexité")


🖼️  EXEMPLES D'ANALYSE GPT-5

1. 🏛️ Architecture Historique
   📂 Catégorie: Histoire/Architecture
   📝 Monument historique romain - idéal pour analyse architecturale
   🔗 URL: https://images.unsplash.com/photo-1539037116277-4db20889f2d4...

2. 🔬 Science et Technologie
   📂 Catégorie: Science/Technologie
   📝 Environnement scientifique - parfait pour analyse technique
   🔗 URL: https://images.unsplash.com/photo-1532094349884-543bc11b234d...

3. 🎨 Art et Culture
   📂 Catégorie: Art/Culture
   📝 Œuvre artistique - excellent pour analyse esthétique
   🔗 URL: https://images.unsplash.com/photo-1541961017774-22349e4a1262...

4. 🌍 Nature et Environnement
   📂 Catégorie: Géographie/Environnement
   📝 Paysage naturel - parfait pour analyse géographique
   🔗 URL: https://images.unsplash.com/photo-1506905925346-21bda4d32df4...

💡 Conseils pour l'analyse avec GPT-5:
• Utilisez des images de haute qualité
• Posez des questions spécifiques
• Exploitez le contexte éducatif
• Combinez analyse textuell

### Test : analyse d'une image de demonstration

Testons la fonction d'analyse sur une image concrete. GPT-5 va decrire le contenu, identifier les éléments visuels, et fournir une analyse structuree.



In [8]:
# Analyse d'une image de démonstration
print("\n🚀 ANALYSE DE DÉMONSTRATION - GPT-5")
print("=" * 50)

# Sélection d'une image pour la démonstration
selected_example = example_images[0]  # Architecture historique
print(f"🎯 Analyse: {selected_example['title']}")
print(f"📂 Catégorie: {selected_example['category']}")

# Test de l'analyse avec les différents modes
analysis_results = []

for mode in ['quick', 'detailed', 'educational']:
    print(f"\n🔍 Mode d'analyse: {mode.upper()}")
    print("-" * 30)
    
    # Analyse de l'image
    result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        analysis_type=mode
    )
    
    if result['success']:
        print(f"✅ Analyse {mode} réussie")
        print(f"⏱️  Temps: {result['metadata']['response_time']:.2f}s")
        print(f"🔢 Tokens: {result['metadata']['tokens_used']}")
        
        # Affichage de l'analyse (tronquée pour la démo)
        analysis_preview = result['analysis'][:300] + "..." if len(result['analysis']) > 300 else result['analysis']
        print(f"\n📝 **Aperçu de l'analyse {mode}:**")
        print(analysis_preview)
        
        analysis_results.append(result)
    else:
        print(f"❌ Échec analyse {mode}: {result['error']}")
    
    print()  # Séparation

# Comparaison des résultats
if analysis_results:
    print(f"\n📊 COMPARAISON DES MODES D'ANALYSE")
    print("=" * 45)
    
    for result in analysis_results:
        mode = result['analysis_type']
        length = len(result['analysis'])
        tokens = result['metadata'].get('tokens_used', {}).get('total_tokens', 'N/A')
        time = result['metadata']['response_time']
        
        print(f"{mode.capitalize():12} | {length:4d} chars | {tokens:>6} tokens | {time:5.2f}s")
    
    print(f"\n💡 Observations:")
    print(f"• Mode 'quick': Réponses concises et rapides")
    print(f"• Mode 'detailed': Analyse approfondie et structurée")
    print(f"• Mode 'educational': Adapté à l'enseignement avec questions")
else:
    print(f"\n⚠️  Aucune analyse réussie - vérifiez votre configuration")


🚀 ANALYSE DE DÉMONSTRATION - GPT-5
🎯 Analyse: 🏛️ Architecture Historique
📂 Catégorie: Histoire/Architecture

🔍 Mode d'analyse: QUICK
------------------------------

🔍 Analyse en cours...
📝 Type: quick
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse quick réussie
⏱️  Temps: 5.36s
🔢 Tokens: {'prompt_tokens': 653, 'completion_tokens': 285, 'total_tokens': 938, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00366625, 'upstream_inference_prompt_cost': 0.00081625, 'upstream_inference_completions_cost': 0.00285}, 'completion_tokens_details': {'reasoning_tokens': 192, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse quick:**
Vue aérienne de la Gran Vía à Madrid au coucher du soleil : immeubles historiques éclairés, dont le dôme noir et doré du bâtiment Metrópolis, trafic léger sur l’avenue, ciel orangé avec les montagnes à l’horizon.


🔍 Mode d'analyse: DETAILED
------------------------------

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse detailed réussie
⏱️  Temps: 24.04s
🔢 Tokens: {'prompt_tokens': 758, 'completion_tokens': 1996, 'total_tokens': 2754, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.0209075, 'upstream_inference_prompt_cost': 0.0009475, 'upstream_inference_completions_cost': 0.01996}, 'completion_tokens_details': {'reasoning_tokens': 1152, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse detailed:**
1) Description générale
- Vue aérienne/prise de hauteur d’un centre-ville dense au coucher du soleil.
- Une large avenue diagonale traverse l’image, bordée d’immeubles haussmanniens/éclectiques illuminés.
- Au coin gauche, un édifice à dôme noir orné d’une statue ailée et d’ornements dorés; sur la d...


🔍 Mode d'analyse: EDUCATIONAL
------------------------------

🔍 Analyse en cours...
📝 Type: educational
🖼️  Source: https://images.unsplash.com/phot

✅ Analyse educational réussie
⏱️  Temps: 28.48s
🔢 Tokens: {'prompt_tokens': 804, 'completion_tokens': 1671, 'total_tokens': 2475, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.017715, 'upstream_inference_prompt_cost': 0.001005, 'upstream_inference_completions_cost': 0.01671}, 'completion_tokens_details': {'reasoning_tokens': 832, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse educational:**
🎯 Analyse pédagogique

1) Description accessible
- Vue urbaine au coucher du soleil, prise en hauteur. Une large avenue traverse la scène entre des immeubles historiques éclairés. On distingue un bâtiment d’angle coiffé d’un dôme sombre et orné de sculptures, des façades blanches alignées, de la cir...


📊 COMPARAISON DES MODES D'ANALYSE
Quick        |  212 chars |    938 tokens |  5.36s
Detailed     | 3214 chars |   2754 tokens | 24.04s
Education

### Lecture du résultat — comparaison quick/detailed/educational

La cellule précédente lance trois appels successifs sur l'image `Architecture Historique` (en fait la Gran Vía à Madrid — première surprise factuelle à noter). La sortie se termine par une table `📊 COMPARAISON DES MODES D'ANALYSE` imprimée par le code lui-même : caractères, tokens et temps y sont récapitulés pour les trois modes — relisez-la directement dans la sortie, elle reflète l'exécution que vous venez de lancer (ses valeurs changent à chaque ré-exécution).

**Ce qu'on y lit, en guide** :
1. Le temps de réponse n'est pas linéaire en longueur de sortie : comparez le rapport des `total_tokens` et celui des temps — la latence est dominée par la phase de raisonnement (`reasoning_tokens`, dans les `completion_tokens_details` de chaque bloc), pas par la génération elle-même.
2. Le coût upstream (`upstream_inference_cost` dans les détails de tokens) suit approximativement les tokens totaux ; le prompt `educational` est le plus cher parce qu'il injecte structure + format de sortie dans l'entrée.
3. Pour un usage `alt-text`, c'est le prompt spécialisé de la section suivante qui garantit la contrainte (≤ 125 caractères imposée dans le prompt) — la sortie `quick` est concise, mais c'est le prompt qui tient la promesse de longueur, pas la seule brièveté du mode. Pour un usage pédagogique structuré, le mode `educational` produit des questions de guidage de l'élève en sus.

### Exemple guidé : un prompt d'analyse personnalisé pour le domaine Art

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20551), intégrée comme exemple guidé.*

Les trois modes d'analyse (quick, detailed, educational) couvrent des usages généraux, mais les cas réels nécessitent souvent des prompts spécialisés. Cet exemple construit un prompt optimisé pour le domaine **Art** : rôle expert, quatre sections imposées (description technique, analyse stylistique, contexte et signification, questions guides), exigence de vocabulaire spécifique, et consigne de factualité qui distingue ce que le modèle voit de ce qu'il déduit.

**Points clés** :
- Le prompt impose le **format de sortie** (sections numérotées) et la **langue** (`language`) — la réponse devient vérifiable mécaniquement
- La qualité n'est pas seulement lue, elle est **mesurée par le code** : présence des sections, vocabulaire artistique détecté, nombre de questions guides
- Une évaluation automatisée transforme une impression (« la réponse est bonne ») en critère reproductible


In [9]:
# Exemple guide : un prompt d'analyse personnalise pour le domaine Art
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# Etape 1 : Choix du domaine
mon_domaine = "Art"  # parmi "Histoire", "Sciences", "Geographie", "Art", "Medecine"

# Etape 2 : Creation du prompt personnalise
# Structure suivie : role expert, format de sortie en sections, vocabulaire
# specifique au domaine, questions guides, consigne de factualite
mon_prompt_analyse = f"""Tu es un professeur expert en {mon_domaine}
Réponds en {language}, avec un vocabulaire précis propre au domaine de l'art.

Structure ta réponse avec exactement ces sections :

**1. Description technique**
- Sujet représenté, type d'image (photographie, peinture, architecture...)
- Composition : cadrage, plans, lignes de force, points focaux
- Palette : couleurs dominantes, contrastes, lumière (source, direction, ambiance)

**2. Analyse stylistique**
- Style ou courant artistique reconnaissable (ex. néoclassique, éclectique, Art déco)
- Éléments formels notables : ornementation, symétrie, rythme, perspective
- Utilise au moins 5 termes de vocabulaire spécifique (ex. façade, corniche, dôme, clair-obscur, point de fuite)

**3. Contexte et signification**
- Période et contexte culturel probables
- Message ou émotion que l'image cherche à transmettre

**4. Trois questions guides pour les étudiants**
- Trois questions numérotées, chacune se terminant par un point d'interrogation
- Progression : une question d'observation, une d'analyse, une d'interprétation

Reste factuel : distingue ce que tu vois de ce que tu déduis. """

# Etape 3 : Test sur l'image de demonstration
image_test = example_images[2]['url']  # Peinture abstraite (Art et Culture)
resultat_exo1 = analyze_image_with_gpt5(image_test, prompt=mon_prompt_analyse)

if resultat_exo1['success']:
    texte = resultat_exo1['analysis']
    print(texte)
    print(f"\nTemps : {resultat_exo1['metadata']['response_time']:.1f}s")
else:
    print(f"Échec : {resultat_exo1['error']}")
    texte = ""

# Etape 4 : Evaluation automatique de la qualite
if texte:
    t = texte.lower()

    # La reponse contient-elle bien les sections demandees ?
    sections = ["description technique", "analyse stylistique",
                "contexte", "questions guides"]
    print("\n--- Sections présentes ---")
    for s in sections:
        print(f"{'OK ' if s in t else 'MANQUE'}  {s}")

    # Le vocabulaire est-il adapte au domaine ?
    vocab_art = ["composition", "palette", "lumière", "perspective", "style",
                 "façade", "ornement", "contraste", "symétrie", "plan",
                 "point de fuite", "ligne", "cadrage", "dôme", "corniche"]
    trouves = [v for v in vocab_art if v in t]
    print(f"\n--- Vocabulaire artistique : {len(trouves)}/{len(vocab_art)} termes ---")
    print(", ".join(trouves))

    # Les questions guides sont-elles presentes (3 attendues) ?
    questions = [l.strip() for l in texte.splitlines() if l.strip().endswith("?")]
    print(f"\n--- Questions guides détectées : {len(questions)} (attendu : 3) ---")
    for q in questions:
        print(" •", q)



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800


1. Description technique
- Sujet représenté, type d'image
  - Sujet: abstraction non figurative, sans motif reconnaissable.
  - Type: peinture sur toile, probable acrylique (coulures nettes, séchage rapide) avec forts empâtements.
- Composition: cadrage, plans, lignes de force, points focaux
  - Cadrage serré, sans marge; la trame de la toile est visible.
  - Organisation en champs verticaux: à gauche un pan bleu-vert, au centre une réserve blanche striée, à droite une masse orangée-rouge; grand aplat jaune en arrière-plan.
  - Lignes de force majoritairement verticales (traînées, dégoulinures/drippings), avec quelques trajectoires obliques qui dynamisent la surface.
  - Points focaux: contrastes durs aux jointures jaune/bleu-vert et orange/blanc; petites éclaboussures rouge carmin attirant l’œil par saturation.
- Palette: couleurs dominantes, contrastes, lumière
  - Dominantes: jaune cadmium, orange brûlé, rouge carmin, bleu-vert/émeraude, rehauts de blanc titane.
  - Contrastes compl

### Applications pratiques : accessibilite et pedagogie

GPT-5 excelle dans deux cas d'usage concrets :
- **Alt text** : generer automatiquement des descriptions d'accessibilite pour les images web
- **Analyse pedagogique** : utiliser la vision multimodale pour créer du contenu educatif a partir d'images


### Lecture du résultat — génération d'alt text

La cellule précédente appelle `analyze_image_with_gpt5()` avec un prompt spécialisé accessibilité qui impose trois contraintes : (1) sortie ≤ 125 caractères, (2) ton factuel et objectif, (3) pas de formatage additionnel. La sortie obtenue décrit la scène visible en une phrase factuelle unique ; le `print` affiche sa longueur en caractères — vérifiez qu'elle reste sous le seuil de 125 imposé par le prompt (le texte exact change à chaque exécution, la contrainte, elle, est tenue par le prompt).

**Pourquoi cette longueur compte** : WCAG 2.1 (critère 1.1.1 *Non-text Content*) exige une alternative textuelle qui rend le **même sens** que l'image — la norme **n'impose aucune longueur**. Le seuil de 125 caractères est une **convention de ce carnet**, inscrite dans le prompt, et non une valeur normative : au-delà, l'alt text devient long à lire en consultation rapide ; en dessous de 50, il perd le contexte. L'exemple guidé de la cellule 19 ci-dessous exécute les trois niveaux de longueur pour étudier cette zone de confort — c'est l'un des usages où la multimodalité native remplace un pipeline séparé (modèle vision → modèle texte → formatage).

In [10]:
# Génération de descriptions d'accessibilité (Alt text)
if generate_alt_text and analysis_results:
    print("\n♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ")
    print("=" * 55)
    
    # Prompt spécialisé pour l'accessibilité
    accessibility_prompt = f"""Génère une description d'accessibilité (alt text) pour cette image en {language}.
    
Critères :
- Maximum 125 caractères
- Description factuelle et objective
- Inclut les éléments essentiels pour la compréhension
- Évite les interprétations subjectives
- Adapté aux lecteurs d'écran

Format : Fournis UNIQUEMENT la description, sans formatage supplémentaire."""
    
    # Génération de l'alt text
    alt_result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        prompt=accessibility_prompt
    )
    
    if alt_result['success']:
        alt_text = alt_result['analysis'].strip()
        
        print(f"✅ Description d'accessibilité générée")
        print(f"📝 Alt text ({len(alt_text)} caractères) :")
        print(f'   "{alt_text}"')
        
        if len(alt_text) > 125:
            print(f"⚠️  Longueur dépassée ({len(alt_text)}/125 chars) - considérez une version plus courte")
        else:
            print(f"✅ Longueur optimale ({len(alt_text)}/125 chars)")
    else:
        print(f"❌ Erreur génération alt text: {alt_result['error']}")
else:
    print(f"\n⏭️  Génération alt text désactivée ou pas d'analyse disponible")


♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Description d'accessibilité générée
📝 Alt text (118 caractères) :
   "Vue en hauteur de la Gran Vía à Madrid au coucher du soleil, avec l'immeuble Metropolis et la circulation sur l'avenue"
✅ Longueur optimale (118/125 chars)


### Exemple guidé : descriptions d'accessibilité multi-niveaux

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20551), intégrée comme exemple guidé.*

L'accessibilité web exige des textes alternatifs (alt text) adaptés au contexte. Une même image peut nécessiter des descriptions de longueurs différentes selon l'usage : navigation rapide (≤ 125 caractères), description détaillée pour lecteurs d'écran (200-400 caractères), ou résumé pédagogique. Cet exemple génère les trois niveaux avec trois prompts distincts — chacun fixe explicitement la longueur cible et le public visé — puis mesure par le code la longueur réelle de chaque texte produit.

**Points clés** :
- Chaque niveau a son **propre prompt** : la longueur maximale et le public sont écrits noir sur blanc dans la consigne
- La vérification s'appuie sur `len()` des textes réels — la conformité à la limite imposée par le prompt est **constatée**, pas supposée
- Le texte court vise ≤ 125 caractères : au-delà, l'alt text devient long à lire en consultation rapide (cf. l'analyse ci-dessus)

In [11]:
# Exemple guide : descriptions d'accessibilite multi-niveaux
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# Etape 1 : Choix de l'image
mon_image = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1BF0cJuaFBV8c&s=10"  # image choisie par le groupe

# Etape 2 : Prompt pour alt text court (<= 125 caracteres)
prompt_court = "Fournissez une description factuelle et objective de l'image, en maximum 125 caractères, sans interprétation."

# Etape 3 : Prompt pour description detaillee (200-400 caracteres)
prompt_detaille = "Fournissez une description détaillée de l'image, en 200 à 400 caractères, pour les lecteurs d'écran. Incluez les éléments essentiels, les couleurs, la disposition et les actions visibles."
# Contraintes imposees : elements essentiels, couleurs, disposition, actions visibles

# Etape 4 : Prompt pour resume pedagogique niveau "secondary"
prompt_pedago = "Fournissez un résumé pédagogique de l'image, adapté à des élèves du secondaire. Utilisez un vocabulaire accessible et posez des questions de réflexion."

# Generation des trois niveaux
resultats = {}
for nom, prompt in [("court", prompt_court), ("detaille", prompt_detaille), ("pedago", prompt_pedago)]:
    r = analyze_image_with_gpt5(mon_image, prompt=prompt)
    if r['success']:
        resultats[nom] = r['analysis']
        print(f"\n--- {nom.upper()} ({len(r['analysis'])} chars) ---")
        print(r['analysis'][:300])

# Etape 5 : Verification des contraintes de longueur
print("\n Vérification des résultats")
print("Nombre de caractères dans le texte court:", len(resultats.get("court", "")))
print("Nombre de caractères dans la description détaillée:", len(resultats.get("detaille", "")))

print("\nRésultats disponibles pour les trois niveaux : court, détaillé et pédagogique:")
print("Texte court (<= 125 chars):", resultats.get("court", "Non généré"))
print("Description détaillée (200-400 chars):", resultats.get("detaille", "Non généré"))
print("Résumé pédagogique:", resultats.get("pedago", "Non généré"))



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- COURT (111 chars) ---
Personnage 3D masculin en tenue sombre, gants noirs, brandissant un pistolet, vu en buste dans un décor sombre.

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- DETAILLE (321 chars) ---
Plan rapproché d’un personnage masculin réaliste type jeu vidéo, cadré buste. Il tient un pistolet noir levé de la main droite gantée, expression dure tournée à gauche. Tenue tactique bleu nuit et noire, muscles saillants, harnais et sangle d’une arme longue. Arrière-plan industriel sombre, éclairag

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSm6AMs5m9c7Nmp-IB1HUY4LeA4EjTF8sUTmW3JU-26cs1B...



--- PEDAGO (2276 chars) ---
Résumé pédagogique
- L’image montre un personnage fictif, créé en images de synthèse (3D). 
- Il porte une tenue tactique sombre, des gants et tient un pistolet. Son regard est sérieux et concentré. 
- Le décor est sombre, peut-être un lieu industriel. La lumière vient de côté et met en valeur son v

 Vérification des résultats
Nombre de caractères dans le texte court: 111
Nombre de caractères dans la description détaillée: 321

Résultats disponibles pour les trois niveaux : court, détaillé et pédagogique:
Texte court (<= 125 chars): Personnage 3D masculin en tenue sombre, gants noirs, brandissant un pistolet, vu en buste dans un décor sombre.
Description détaillée (200-400 chars): Plan rapproché d’un personnage masculin réaliste type jeu vidéo, cadré buste. Il tient un pistolet noir levé de la main droite gantée, expression dure tournée à gauche. Tenue tactique bleu nuit et noire, muscles saillants, harnais et sangle d’une arme longue. Arrière-plan industr

Les analyses de demonstration sont completes. Le mode interactif ci-dessous permet d'experimenter avec vos propres images et prompts d'analyse personnalises.


### Lecture du résultat — mode batch vs interactif

La cellule précédente tente `input()` pour récupérer une URL ou un chemin local. En exécution Papermill / batch, l'entrée standard n'est pas disponible : le `except EOFError` intercepte proprement et bascule sur le message `⏭️  Mode interactif non disponible (exécution automatisée)`. Ce pattern est la convention du dépôt pour les notebooks qui mélangent usage interactif et usage batch : la cellule reste fonctionnelle dans les deux modes sans `raise NotImplementedError` (interdit par C.1 — `pass`/`print` sont les stubs corrects).

La cellule suivante passe aux cas d'usage applicatifs — c'est le pont entre la démonstration technique et l'usage métier. Les 5 domaines listés (Histoire, Sciences, Géographie, Art, Médecine) couvrent les disciplines où l'analyse multimodale apporte le plus vs un modèle purement textuel.

In [12]:
# Mode interactif - Analyse d'image personnalisée
if notebook_mode == "interactive" and not skip_widgets:
    print("\n🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE")
    print("=" * 55)
    
    print("\n💡 Analysez votre propre image avec GPT-5:")
    print("Formats supportés: URL https:// ou chemin local")
    print("(Laissez vide pour passer à la suite)")
    
    try:
        user_image = input("\n🖼️  URL ou chemin de votre image: ").strip()

        if user_image:
            # Paramètres d'analyse personnalisés
            print("\n⚙️  Paramètres d'analyse (appuyez Entrée pour défaut):")
            custom_mode = input(f"📊 Mode [{analysis_mode}]: ").strip() or analysis_mode
            custom_prompt = input("📝 Prompt personnalisé (optionnel): ").strip()

            print(f"\n🔍 Analyse de votre image en cours...")

            # Analyse personnalisée
            if custom_prompt:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    prompt=custom_prompt
                )
            else:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    analysis_type=custom_mode
                )

            if user_result['success']:
                print(f"\n🎉 Analyse réussie!")
                print(f"⏱️  Temps: {user_result['metadata']['response_time']:.2f}s")
                print(f"\n📝 **Analyse GPT-5:**")
                print(user_result['analysis'])

                # Option de sauvegarde
                if export_analysis:
                    try:
                        save_choice = input("\n💾 Sauvegarder cette analyse ? (o/N): ").strip().lower()
                        if save_choice in ['o', 'oui', 'y', 'yes']:
                            output_dir = current_path / 'outputs' / 'gpt5_analysis'
                            output_dir.mkdir(parents=True, exist_ok=True)

                            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                            analysis_file = output_dir / f"gpt5_analysis_{timestamp}.json"

                            with open(analysis_file, 'w', encoding='utf-8') as f:
                                json.dump(user_result, f, indent=2, ensure_ascii=False)

                            print(f"💾 Analyse sauvegardée: {analysis_file}")
                    except Exception as save_err:
                        if "StdinNotImplemented" in type(save_err).__name__:
                            print("\n⏭️  Sauvegarde automatique ignorée en mode batch")
                        else:
                            raise
            else:
                print(f"\n❌ Erreur: {user_result['error']}")
                print(f"🔍 Source: {user_result['image_source']}")
        else:
            print("\n⏭️  Mode interactif ignoré")

    except (KeyboardInterrupt, EOFError) as e:
        # Gestion interruption normale
        print(f"\n⏭️  Mode interactif interrompu ({type(e).__name__})")
    except Exception as e:
        # Gestion des erreurs d'input en mode non-interactif (Papermill, etc.)
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("\n⏭️  Mode interactif non disponible (exécution automatisée)")
        else:
            # Autre erreur inattendue - afficher pour débogage
            print(f"\n⚠️  Erreur inattendue: {error_type} - {str(e)[:100]}")
            print("⏭️  Passage à la suite du notebook")
else:
    print("\n🤖 Mode batch - Interface interactive désactivée")
    print("💡 Pour mode interactif: notebook_mode = 'interactive'")


🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE

💡 Analysez votre propre image avec GPT-5:
Formats supportés: URL https:// ou chemin local
(Laissez vide pour passer à la suite)

⏭️  Mode interactif non disponible (exécution automatisée)


Le mode interactif etant traite, la cellule suivante presente les principaux cas d'usage applicatifs de l'analyse visuelle par GPT-5 dans un contexte professionnel et pedagogique.


### Lecture du résultat — matrice domaine × prompt

La cellule précédente catalogue cinq domaines d'application (Histoire, Sciences, Géographie, Art et Culture, Médecine) avec pour chacun un `prompt_template` prédéfini qui peut être passé directement à `analyze_image_with_gpt5()`. Chaque template impose un rôle disciplinaire, un format de sortie, et un ton adapté au domaine — par exemple, le prompt `Médecine` spécifie *Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique.*

L'output ne contient pas de sortie GPT-5 visible — la cellule a uniquement *annoncé* les cas d'usage, sans encore les exécuter. L'exemple guidé qui suit en exécute un pour la géographie, et les exercices de fin de notebook réutilisent la même fonction d'analyse sur d'autres questions : stabilité d'une sortie répétée, fidélité d'un alt text, notation structurée en JSON. C'est une inversion pédagogique intéressante : la cellule 24 montre les templates, les exercices demandent d'appliquer la fonction d'analyse à d'autres questions.

In [13]:
# Cas d'usage pédagogiques avec GPT-5
print("\n🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5")
print("=" * 45)

# Exemples d'applications éducatives
educational_use_cases = {
    "Histoire": {
        "description": "Analyse de documents historiques, œuvres d'art, monuments",
        "exemple": "Analyser une fresque renaissance pour comprendre le contexte social",
        "prompt_template": "Analyse cette image historique pour des étudiants en histoire. Explique le contexte historique, les éléments symboliques, et l'importance culturelle."
    },
    
    "Sciences": {
        "description": "Analyse d'expériences, schémas scientifiques, phénomènes naturels",
        "exemple": "Expliquer un diagramme de cellule ou une réaction chimique",
        "prompt_template": "En tant qu'enseignant de sciences, explique cette image scientifique. Identifie les éléments techniques et leur fonctionnement."
    },
    
    "Géographie": {
        "description": "Étude de paysages, cartes, phénomènes géologiques",
        "exemple": "Analyser une photo satellite ou un paysage géographique",
        "prompt_template": "Analyse cette image géographique pour des étudiants. Explique les formations géologiques, le climat, et l'impact humain visible."
    },
    
    "Art et Culture": {
        "description": "Critique artistique, analyse stylistique, histoire de l'art",
        "exemple": "Comprendre les techniques d'un tableau impressionniste",
        "prompt_template": "Fais une analyse artistique de cette œuvre pour des étudiants en art. Inclus style, techniques, et signification culturelle."
    },
    
    "Médecine": {
        "description": "Analyse d'imagerie médicale, anatomie, cas cliniques",
        "exemple": "Expliquer une radiographie ou un schéma anatomique",
        "prompt_template": "Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique."
    }
}

# Affichage des cas d'usage
for domain, info in educational_use_cases.items():
    print(f"\n📚 **{domain}**")
    print(f"   📋 {info['description']}")
    print(f"   💡 Exemple: {info['exemple']}")
    print(f"   📝 Template de prompt disponible")

print(f"\n🎯 Avantages GPT-5 pour l'éducation:")
print(f"• **Multimodal** : Analyse texte + image simultanée")
print(f"• **Contextuel** : Comprend le niveau éducatif")
print(f"• **Adaptable** : Ajuste le vocabulaire selon l'audience")
print(f"• **Interactif** : Permet les questions de suivi")
print(f"• **Précis** : Analyse détaillée et factuelle")

print(f"\n📝 Exemple de workflow pédagogique:")
print(f"1. Sélection d'image pertinente au cours")
print(f"2. Analyse GPT-5 avec prompt éducatif")
print(f"3. Génération de questions pour les étudiants")
print(f"4. Discussion interactive basée sur l'analyse")
print(f"5. Évaluation de la compréhension")


🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5

📚 **Histoire**
   📋 Analyse de documents historiques, œuvres d'art, monuments
   💡 Exemple: Analyser une fresque renaissance pour comprendre le contexte social
   📝 Template de prompt disponible

📚 **Sciences**
   📋 Analyse d'expériences, schémas scientifiques, phénomènes naturels
   💡 Exemple: Expliquer un diagramme de cellule ou une réaction chimique
   📝 Template de prompt disponible

📚 **Géographie**
   📋 Étude de paysages, cartes, phénomènes géologiques
   💡 Exemple: Analyser une photo satellite ou un paysage géographique
   📝 Template de prompt disponible

📚 **Art et Culture**
   📋 Critique artistique, analyse stylistique, histoire de l'art
   💡 Exemple: Comprendre les techniques d'un tableau impressionniste
   📝 Template de prompt disponible

📚 **Médecine**
   📋 Analyse d'imagerie médicale, anatomie, cas cliniques
   💡 Exemple: Expliquer une radiographie ou un schéma anatomique
   📝 Template de prompt disponible

🎯 Avantages GPT-5 pour l'éducat

### Exemple guidé : analyse personnalisée à trois modes pour un domaine

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20551), intégrée comme exemple guidé.*

Synthèse des deux exemples précédents : la fonction `analyse_complete()` enchaîne trois analyses de la même image — description factuelle courte, description détaillée pour lecteurs d'écran, résumé pédagogique contextualisé par le domaine — et retourne l'ensemble dans un dictionnaire affiché avec `pprint`. Le domaine (`Géographie`), le niveau (`university`) et l'image sont paramétrés en tête de cellule.

**Points clés** :
- Un seul appel de fonction produit les **trois niveaux** de description — le résultat est un dictionnaire réutilisable, pas trois affichages séparés
- Le prompt éducatif est **construit à partir du domaine** : changer `domain` en tête de cellule adapte l'analyse à une autre discipline


In [14]:
# Exemple guide : analyse personnalisee a trois modes pour un domaine
# (contribution d'un groupe d'etudiants EPF, TP2, PR #20551)

# 1. Sélection du domaine et configuration
domain = "Géographie"
niveau = "university"  # Niveau éducatif cible

# 2. Image à analyser (URL publique)
image_source = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s=10"  # carte de France (choix du groupe)

# 3. Création du prompt personnalisé (inspiré des prompt_template de educational_use_cases)
custom_prompt = f"""Tu es un professeur expert en {domain}.
Analyse cette image pour des étudiants de niveau {niveau}.
Inclus :
1. Description accessible du sujet
2. Éléments techniques importants à observer
3. Contexte historique ou scientifique
4. 3 questions de réflexion pour les étudiants
"""

# 4. Analyse avec les 3 modes de description
def analyse_complete(image_url, prompt, domain):
    """
    Analyse une image avec les 3 modes et retourne les résultats.

    Args:
        image_url: URL ou chemin de l'image
        prompt: Prompt personnalisé pour le mode educational
        domain: Domaine d'étude pour le contexte

    Returns:
        Dict avec les 3 analyses (quick, detailed, educational)
    """



    quick = "Fournissez une description factuelle et objective de l'image, en maximum 125 caractères, sans interprétation."
    detailed = "Fournissez une description détaillée de l'image, en 200 à 400 caractères, pour les lecteurs d'écran. Incluez les éléments essentiels, les couleurs, la disposition et les actions visibles."
    educational = f"""Fournissez un résumé pédagogique de l'image, adapté à des élèves du {domain}. Utilisez un vocabulaire accessible et posez des questions de réflexion."""

    results = {}
    for nom, prompt in [("quick", quick), ("detailed", detailed), ("educational", educational)]:
        r = analyze_image_with_gpt5(image_source, prompt=prompt)
        if r['success']:
          results[nom] = r['analysis']
    return results

# 5. Exécuter l'analyse et afficher les résultats
import pprint
results = analyse_complete(image_source, custom_prompt, domain)
pprint.pprint(results)



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...


{'detailed': 'Carte politique de la France métropolitaine en tons pastel: '
             'anciennes régions délimitées en orange et nommées, avec '
             'principales villes et cours d’eau. Mers en bleu: Manche au '
             'nord‑ouest, Atlantique à l’ouest, Méditerranée au sud. '
             'Frontières avec Espagne, Andorre, Belgique, Luxembourg, '
             'Allemagne, Suisse, Italie. Vignettes: zoom Paris à gauche, Corse '
             'en bas à droite; échelle en km.',
 'educational': 'Résumé pédagogique de l’image\n'
                '- C’est une carte de la France métropolitaine. On y voit les '
                'frontières avec d’autres pays, les mers, les grandes villes, '
                'les montagnes, les fleuves et les anciennes régions '
                'administratives.\n'
                '- Les mers: la Manche au nord, l’océan Atlantique à l’ouest, '
                'la mer Méditerranée au sud-est. L’île de Corse se trouve en '
                'Méditerrané

## 🎯 Résumé et Bonnes Pratiques

### ✅ Ce que vous avez appris

- [ ] **Configuration GPT-5** : OpenRouter API et paramètres optimaux
- [ ] **Analyse multimodale** : Combinaison texte + image
- [ ] **Prompts éducatifs** : Adaptation au niveau d'enseignement
- [ ] **Cas d'usage pédagogiques** : Applications concrètes par domaine
- [ ] **Optimisation** : Paramètres de qualité et performance

### 🚀 Prochaines étapes

1. **Expérimentez** avec vos propres images éducatives
2. **Testez** différents niveaux d'analyse selon votre public
3. **Intégrez** GPT-5 dans vos workflows pédagogiques
4. **Combinez** avec DALL-E 3 pour génération + analyse
5. **Explorez** les notebooks avancés (Module 02)

### 💡 Conseils pour l'utilisation

**✅ Bonnes pratiques:**
- Utilisez des images de haute qualité
- Adaptez le prompt au niveau éducatif
- Combinez analyse visuelle et contextuelle
- Générez des questions pédagogiques
- Sauvegardez les analyses réussies

**❌ Évitez:**
- Images trop petites ou de mauvaise qualité
- Prompts vagues ou génériques
- Oublier le contexte éducatif
- Ignorer les limitations du modèle
- Ne pas vérifier la factualité

### 🔗 Ressources complémentaires

- **Documentation OpenRouter** : [openrouter.ai](https://openrouter.ai)
- **Guide GPT-5** : Capacités multimodales
- **Templates éducatifs** : `docs/genai-phase2-templates.md`
- **Standards CoursIA** : `docs/genai-images-development-standards.md`

### 📖 Références savantes

- **OpenAI** (2025). *GPT-5 System Card & Model Page*. — Modèle multimodal (texte + vision), fenêtre de contexte **400 000 tokens**, 128 000 tokens de sortie maximum. Sources officielles : openai.com/gpt-5/ et developers.openai.com/api/docs/models/gpt-5.
- **OpenAI** (2024). *GPT-4o System Card*. — Précurseur multimodal de GPT-5 (vision + texte unifiés), base de l'API multimodale utilisée via OpenRouter dans ce notebook.


### Lecture du résultat — ce que la cellule suivante présente vraiment

Le `print('Cas d'usage pédagogiques')` précédent est un **catalogue statique**, pas une exécution. Les cinq domaines listés sont là pour donner à l'étudiant un point de départ structuré — chaque `prompt_template` est immédiatement réutilisable via `analyze_image_with_gpt5(image, prompt=...)`, sans avoir à le réécrire. Le `print('Template de prompt disponible')` est l'indice que la cellule a déjà joué son rôle pédagogique au moment où l'étudiant l'a lue, même si rien n'a été exécuté.

**Trois choses à retenir pour la cellule de résumé** :
1. L'API OpenRouter expose une large famille de modèles GPT-5 (l'inventaire de l'exécution est dans la sortie de connexion) — la nomenclature est plus large que la nomenclature OpenAI officielle, et la sélection se fait sur le compromis `contexte × coût × capabilities` (vision, tools, etc.).
2. Les trois modes d'analyse (quick/detailed/educational) ne sont pas interchangeables : quick est optimisé pour l'alt-text court, educational produit des questions de guidage de l'élève — c'est le seul mode qui fournit un livrable pédagogique clé en main.
3. Le pipeline multimodal (encode → POST → parse) est uniforme quelle que soit l'image — la complexité se trouve dans le prompt, pas dans le transport.

***

## Exemples guidés : trois gestes mesurés

Trois exemples guidés issus d'une contribution étudiante closent la partie « analyse » du carnet : mesurer la sensibilité d'une analyse à la formulation du prompt (similarité lexicale entre deux sorties à format constant), adapter un alt text à son contexte de publication sous une limite de longueur vérifiée par mesure, et comparer deux images d'un même domaine avec un prompt d'analyse identique. Le code est conservé tel qu'il a été rendu — il s'exécute et imprime ses propres contrôles.

### Exemple guidé : prompt neutre vs prompt orienté — mesurer la sensibilité de la formulation

*Contribution étudiante (PR #23305, github.com/Perr1neF0), intégrée comme exemple guidé.*

Une même image analysée par GPT-5 peut produire des réponses très différentes selon la formulation du prompt. L'exemple mesure cette sensibilité au lieu de la subir : deux prompts qui ne diffèrent que par leur **angle** (neutre vs orienté vers l'histoire des sciences), même image, même format de sortie imposé.

**Points clés** :
- Le format de sortie est factorisé dans une seule variable (`format_sortie`) — si le format différait aussi, la comparaison ne mesurerait plus la formulation
- La comparaison est **chiffrée** : un Jaccard lexical sur les mots de cinq lettres et plus, calculé par `mots_cles()`, remplace l'estimation à l'œil
- Un contrôle vérifie que les trois sections demandées sont présentes dans chaque sortie ; sans lui, un écart pourrait n'être qu'un non-respect du format
- La conclusion imprimée relie chaque formulation à un usage : inventaire visuel d'un côté, cours disciplinaire de l'autre

In [15]:
# Exemple guide : prompt neutre vs prompt oriente -- mesurer la sensibilite de la formulation
# (contribution etudiante, PR #23305 -- github.com/Perr1neF0)
import re

# Etape 1 : Choix de l'image
image_exo1 = example_images[1]['url']  # Laboratoire (Science et Technologie)

# Etape 2 : Redaction des deux prompts
format_sortie = f"""Réponds en {language}, avec exactement ces 3 sections :
**1. Éléments visibles** : liste à puces (5 à 8 éléments)
**2. Analyse** : 5 lignes maximum
**3. Conclusion** : une seule phrase"""
prompt_neutre = (
    "Décris cette image de façon objective, sans cadre théorique particulier.\n\n"
    + format_sortie
)
prompt_oriente = (
    "Analyse cette image sous l'angle de l'histoire des sciences : "
    "instruments, méthodes de travail et évolution des pratiques de laboratoire.\n\n"
    + format_sortie
)
# Indice : seul l'angle change -- le format de sortie reste identique

# Etape 3 : Lancement des deux analyses
resultat_neutre = analyze_image_with_gpt5(image_exo1, prompt=prompt_neutre)
resultat_oriente = analyze_image_with_gpt5(image_exo1, prompt=prompt_oriente)

# Etape 4 : Comparaison mesuree
def mots_cles(texte, taille_min=5):
    """Ensemble des mots significatifs (hors mots courts) d'un texte."""
    return {m for m in re.findall(r"[a-zàâçéèêëîïôûùüÿœ'-]+", texte.lower())
            if len(m) >= taille_min}

if resultat_neutre['success'] and resultat_oriente['success']:
    t_neutre = resultat_neutre['analysis']
    t_oriente = resultat_oriente['analysis']
    m_neutre, m_oriente = mots_cles(t_neutre), mots_cles(t_oriente)

    print(f"Longueur : neutre={len(t_neutre)} car. | orienté={len(t_oriente)} car.")
    print(f"Mots communs : {len(m_neutre & m_oriente)}")
    print(f"Jaccard (similarité lexicale) : "
          f"{len(m_neutre & m_oriente) / len(m_neutre | m_oriente):.2f}")
    print("\nSeulement dans le NEUTRE  :", sorted(m_neutre - m_oriente)[:15])
    print("Seulement dans l'ORIENTÉ  :", sorted(m_oriente - m_neutre)[:15])

    # Sections respectees ? (le format doit etre identique pour que la comparaison soit valide)
    for nom, t in [("neutre", t_neutre), ("orienté", t_oriente)]:
        ok = all(s in t.lower() for s in ["éléments visibles", "analyse", "conclusion"])
        print(f"Format respecté ({nom}) : {'OUI' if ok else 'NON'}")

    print("\n--- NEUTRE ---\n", t_neutre)
    print("\n--- ORIENTÉ ---\n", t_oriente)

    print("\nConclusion : le prompt neutre convient à un inventaire visuel (observation, "
          "alt text, premier contact avec l'image) ; le prompt orienté convient à un "
          "cours disciplinaire, car il fait émerger interprétation et vocabulaire du cadre choisi.")
else:
    for nom, r in [("neutre", resultat_neutre), ("orienté", resultat_oriente)]:
        if not r['success']:
            print(f"Échec ({nom}) : {r['error']}")


🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800


Longueur : neutre=959 car. | orienté=1254 car.
Mots communs : 20
Jaccard (similarité lexicale) : 0.15

Seulement dans le NEUTRE  : ['accent', 'atténue', 'autres', 'brillante', 'cadrage', 'central', 'champ', 'contenant', 'cours', 'droite', 'désaturés', 'faible', 'focal', 'froide', 'froids']
Seulement dans l'ORIENTÉ  : ['absence', 'amélioration', 'appareillage', 'après', 'asepsie', 'automatisation', 'borosilicate', 'borosilicaté', 'béchers', 'capacités', 'capteurs', 'chimie', 'chimique', 'condense', 'conique']
Format respecté (neutre) : OUI
Format respecté (orienté) : OUI

--- NEUTRE ---
 **1. Éléments visibles** :
- Un bécher en verre au premier plan contenant un liquide clair légèrement teinté
- Une pipette en plastique insérée dans ce bécher
- Un erlenmeyer en verre vide, posé sur la paillasse, à gauche
- Deux autres récipients en verre transparents à droite, vides
- Une surface de travail blanche et brillante
- Reflets et transparences marqués sur le verre
- Arrière-plan flou, tons g

### Exemple guidé : alt text contextuel — une image, trois contextes

*Contribution étudiante (PR #23305, github.com/Perr1neF0), intégrée comme exemple guidé.*

L'exemple guidé d'accessibilité a montré trois **longueurs** de description pour une même image. Ici, c'est le **contexte de publication** qui change à longueur constante : un alt text pertinent dépend de ce que le lecteur est venu chercher.

**Points clés** :
- Les trois consignes de contexte vivent dans un dictionnaire `contextes`, la contrainte de longueur est partagée (`LIMITE = 125`)
- La contrainte est **vérifiée par mesure** (`len(texte)`), pas à l'œil, avec une relance qui durcit la consigne quand la sortie dépasse la limite
- Deux contrôles terminaux : chaque texte tient sous la limite, et les trois textes sont **distincts** — sans le second, trois reformulations interchangeables passeraient le test

In [16]:
# Exemple guide : alt text contextuel -- une image, trois contextes
# (contribution etudiante, PR #23305 -- github.com/Perr1neF0)

# Etape 1 : Choix de l'image
image_exo2 = example_images[3]['url']  # Paysage naturel

# Etape 2 : Definition des trois contextes de publication
contextes = {
    "article de vulgarisation scientifique": (
        "Contexte : article de vulgarisation scientifique pour un public curieux "
        "mais non spécialiste. Privilégie le phénomène naturel ou géologique visible "
        "(relief, eau, végétation, climat)."
    ),
    "publication sur un reseau social": (
        "Contexte : publication sur un réseau social. Privilégie l'ambiance, la "
        "lumière et l'impression d'ensemble, avec un ton vivant mais factuel."
    ),
    "manuel scolaire de geographie": (
        "Contexte : manuel scolaire de géographie pour des élèves. Privilégie le "
        "vocabulaire géographique précis (type de paysage, composantes, échelle)."
    ),
}

# Etape 3 : Redaction d'un prompt par contexte
LIMITE = 125

def prompt_alt(consigne_contexte, limite=LIMITE):
    return (
        f"Génère un alt text en {language} pour cette image.\n"
        f"{consigne_contexte}\n"
        f"Contraintes : {limite} caractères MAXIMUM (espaces compris), "
        "description factuelle, une seule phrase, aucune interprétation subjective.\n"
        "Réponds UNIQUEMENT avec le texte, sans guillemets ni préambule."
    )

def generer_alt_text(consigne_contexte, tentatives=3):
    """Appelle GPT-5 et relance si la limite de 125 caracteres est depassee."""
    prompt = prompt_alt(consigne_contexte)
    texte = ""
    for _ in range(tentatives):
        r = analyze_image_with_gpt5(image_exo2, prompt=prompt)
        if not r['success']:
            return f"ERREUR : {r['error']}"
        texte = r['analysis'].strip().strip('"\'«» ')
        if len(texte) <= LIMITE:
            return texte
        # On renforce la contrainte avec la longueur reelle constatee
        prompt = prompt_alt(consigne_contexte, limite=LIMITE - 20) + (
            f"\nTa version précédente faisait {len(texte)} caractères : raccourcis-la."
        )
    return texte  # derniere version, meme si trop longue (signalee plus bas)

# Etape 4 : Generation et verification
alt_texts = {ctx: generer_alt_text(consigne) for ctx, consigne in contextes.items()}

print("--- Alt texts générés ---")
for ctx, texte in alt_texts.items():
    statut = "OK" if len(texte) <= LIMITE else "TROP LONG"
    print(f"[{statut}] ({len(texte):3d} car.) {ctx}\n    -> {texte}\n")

# Les trois textes doivent etre distincts
textes = [t.lower() for t in alt_texts.values()]
print("Trois textes distincts :", "OUI" if len(set(textes)) == len(textes) else "NON")
print("Tous <= 125 caractères  :", "OUI" if all(len(t) <= LIMITE for t in alt_texts.values()) else "NON")


🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800


--- Alt texts générés ---
[OK] (107 car.) article de vulgarisation scientifique
    -> Chaîne de montagnes alpines avec pics enneigés, vallée remplie d’une mer de nuages sous ciel crépusculaire.

[OK] (101 car.) publication sur un reseau social
    -> Chaîne de montagnes enneigées surplombant une mer de nuages, lumière rosée et ciel clair à l’horizon.

[OK] (120 car.) manuel scolaire de geographie
    -> Massif de haute montagne, sommets enneigés et versants rocheux, vallée comblée par une mer de nuages; échelle paysagère.

Trois textes distincts : OUI
Tous <= 125 caractères  : OUI


### Exemple guidé : comparateur d'images — deux images du même domaine côte à côte

*Contribution étudiante (PR #23305, github.com/Perr1neF0), intégrée comme exemple guidé.*

Comparer deux images d'un même domaine en un seul workflow : analyse individuelle de chaque image, puis croisement des deux analyses.

**Points clés** :
- Le prompt d'analyse est **le même** pour les deux images — c'est la condition qui rend la comparaison mécanique plutôt qu'arbitraire
- Le croisement est un troisième appel qui reçoit les deux analyses et **joint l'image B**, pour que le modèle vérifie ses affirmations sur l'image visible
- Une analyse en échec interrompt le croisement (`return` immédiat) : croiser une analyse absente produirait un verdict inventé
- Le temps total est mesuré et renvoyé avec le résultat


In [17]:
# Exemple guide : comparateur d'images -- deux images du meme domaine cote a cote
# (contribution etudiante, PR #23305 -- github.com/Perr1neF0)
import pprint, time

# 1. Selection des deux images
domaine = "Géographie"
image_a = example_images[3]['url']  # Paysage naturel (Géographie/Environnement)
image_b = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s=10"  # carte de France (image deja utilisee plus haut)

# 2. Prompt d'analyse pour chaque image
prompt_analyse = f"""Tu es professeur de {domaine}. Réponds en {language}.
Analyse cette image avec exactement cette structure :
**Type d'image** : (photo, carte, schéma...) en une ligne
**Éléments principaux** : 3 à 5 puces
**Information géographique** : 3 lignes maximum
**Intérêt pédagogique** : une phrase"""

# 3. Fonction de comparaison
def compare_images(url_a, url_b, prompt):
    """
    Analyse deux images du meme domaine et produit une comparaison structuree.

    Returns:
        Dict avec les deux analyses, leur comparaison et le temps total.
    """
    debut = time.time()
    comparaison = {}

    # a) Analyse individuelle, meme prompt pour les deux images
    for cle, url in [("analyse_a", url_a), ("analyse_b", url_b)]:
        r = analyze_image_with_gpt5(url, prompt=prompt)
        if not r['success']:
            comparaison[cle] = None
            comparaison["erreur"] = f"{cle} : {r['error']}"
            return comparaison  # inutile de croiser si une analyse manque
        comparaison[cle] = r['analysis']

    # b) Croisement : on injecte les deux analyses dans le prompt, et on joint
    #    l'image B pour que GPT-5 puisse verifier les affirmations sur l'image visible
    prompt_croisement = f"""Tu compares deux images de {domaine}. Réponds en {language}.
Voici l'analyse de l'image A :
\"\"\"{comparaison['analyse_a']}\"\"\"

Voici l'analyse de l'image B (l'image jointe) :
\"\"\"{comparaison['analyse_b']}\"\"\"

Produis exactement ces trois sections :
**Points communs** : 3 puces
**Différences** : 3 puces
**Verdict** : laquelle des deux images est la plus utile pour introduire le domaine à des étudiants, et pourquoi (3 lignes maximum)."""

    r = analyze_image_with_gpt5(url_b, prompt=prompt_croisement)
    comparaison["comparaison"] = r['analysis'] if r['success'] else f"ERREUR : {r['error']}"
    comparaison["temps_total_s"] = round(time.time() - debut, 1)
    return comparaison

# 4. Executer et afficher
resultat_comparaison = compare_images(image_a, image_b, prompt_analyse)
pprint.pprint(resultat_comparaison, width=100)


🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...



🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPQ_gGhRNQdk5gxBqyA9diGOVps2ZliOpWX2ifjfPSPg&s...


{'analyse_a': "**Type d'image** : Photo de paysage de montagne au crépuscule\n"
              '\n'
              '**Éléments principaux** :\n'
              '- Chaîne de montagnes élevées aux sommets enneigés\n'
              '- Vallée encaissée remplie d’une mer de nuages (stratus)\n'
              '- Crêtes et pentes rocheuses au premier plan\n'
              '- Ciel aux teintes rosées/orangées de lever ou coucher de soleil\n'
              '\n'
              '**Information géographique** :\n'
              'La mer de nuages provient d’une inversion thermique où l’air froid et humide stagne '
              'en vallée sous une couche d’air plus chaud.  \n'
              'Le relief alpin induit un effet orographique, favorisant la formation de nuages bas '
              'dans les fonds de vallée.  \n'
              'Les sommets émergent au-dessus des nuages, illustrant un fort gradient altitudinal '
              'et des milieux climatiques distincts.\n'
              '\n'
            

## Comparaison directe : le modèle compare lui-même les deux images

*Exemple guidé ajouté après la présentation de la contribution #23305, qui reste intacte : il reprend son comparateur d'images et le mesure contre une autre façon de faire.*

L'exemple précédent compare **deux textes**. Le modèle ne voit qu'**une** des deux images au moment où il conclut : le troisième appel reçoit les deux analyses en prose, et seulement l'image B jointe. Tout ce que la description de l'image A a omis, déformé ou inventé passe donc dans la comparaison sans contrôle possible, et les deux images ne sont pas traitées de la même façon.

Les modèles de vision acceptent **plusieurs images dans un même message** : un tableau de parties `image_url` dans `content`. Le modèle regarde alors les deux images lui-même, et il n'y a plus de description intermédiaire où l'information se perd.

**Points clés** :
- **Vérifier, ne pas supposer** : la cellule commence par un appel réel et échoue avec un message explicite si le modèle refuse deux images. Un modèle multimodal annoncé n'est pas un modèle multimodal mesuré.
- **La mesure qui tranche** est structurelle : combien d'images l'appel de comparaison reçoit-il réellement ? La voie « descriptions » en joint **une**, la voie directe **deux**.
- Les deux voies reçoivent **la même consigne de comparaison** ; ce qui change est ce qu'elles ont sous les yeux, pas ce qu'on leur demande.
- Le nombre d'appels, les tokens d'entrée et le temps sont relevés pour les deux voies, sur la même paire d'images.
- Un modèle plus léger de la liste du module suffirait : la comparaison directe est **un** appel, pas trois. Non mesuré ici — les cinq appels de la section ont tourné sur `openai/gpt-5`.

In [18]:
# Exemple guide : le modele compare lui-meme -- un seul appel, les deux images
# Mesure face a la version "descriptions" de l'exemple precedent.
import time


def appel_vision(parts_images, prompt, max_tokens_appel=None):
    """Un appel de vision brut : renvoie le texte, l'usage et la duree.

    Les quatre appels de la comparaison passent par cette meme fonction, donc
    par des requetes identiques en tout point sauf le contenu joint : c'est ce
    qui rend la mesure comparable d'une voie a l'autre.
    """
    payload = {
        "model": model_name,
        "messages": [{
            "role": "user",
            "content": [{"type": "text", "text": prompt}] + parts_images,
        }],
        "max_tokens": max_tokens_appel or max_tokens,
        "temperature": temperature,
        "top_p": top_p,
    }
    debut = time.time()
    response = requests.post(f"{api_base_url}/chat/completions", headers=headers,
                             json=payload, timeout=120)
    duree = round(time.time() - debut, 1)
    if response.status_code != 200:
        try:
            message = response.json().get("error", {}).get("message", "")
        except ValueError:
            message = ""
        return {"ok": False, "erreur": message or f"HTTP {response.status_code}",
                "parts_images": len(parts_images), "secondes": duree, "usage": {}}
    data = response.json()
    return {"ok": True,
            "texte": data["choices"][0]["message"]["content"],
            "usage": data.get("usage", {}),
            "parts_images": len(parts_images),
            "secondes": duree}


def bilan(appels, appel_comparaison):
    """Agrege ce qui est comparable entre les deux voies."""
    return {
        "appels": len(appels),
        "images_vues_a_la_comparaison": appel_comparaison["parts_images"],
        "tokens_entree": sum(int(a["usage"].get("prompt_tokens") or 0) for a in appels),
        "tokens_sortie": sum(int(a["usage"].get("completion_tokens") or 0) for a in appels),
        "secondes": round(sum(a["secondes"] for a in appels), 1),
    }


def comparer_par_descriptions(url_a, url_b, prompt_analyse, prompt_comparaison):
    """Voie de l'exemple precedent : deux descriptions en texte, puis un croisement."""
    image_a_part = prepare_image_for_gpt5(url_a)
    image_b_part = prepare_image_for_gpt5(url_b)
    analyses = {}
    appels = []
    for cle, part in (("a", image_a_part), ("b", image_b_part)):
        r = appel_vision([part], prompt_analyse)
        appels.append(r)
        if not r["ok"]:
            raise RuntimeError(f"analyse de l'image {cle.upper()} : {r['erreur']}")
        analyses[cle] = r["texte"]
    prompt = (f"{prompt_comparaison}\n\n"
              f"Analyse de l'image A :\n\"\"\"{analyses['a']}\"\"\"\n\n"
              f"Analyse de l'image B (image jointe) :\n\"\"\"{analyses['b']}\"\"\"")
    # Une seule image est jointe ici : c'est l'asymetrie que l'on mesure.
    croisement = appel_vision([image_b_part], prompt)
    appels.append(croisement)
    if not croisement["ok"]:
        raise RuntimeError(f"croisement : {croisement['erreur']}")
    return croisement["texte"], bilan(appels, croisement)


def comparer_directement(urls, prompt_comparaison):
    """Voie directe : toutes les images dans un seul message, aucune prose intermediaire."""
    parts_images = [prepare_image_for_gpt5(u) for u in urls]
    comparaison = appel_vision(parts_images, prompt_comparaison)
    if not comparaison["ok"]:
        raise RuntimeError(f"comparaison directe : {comparaison['erreur']}")
    return comparaison["texte"], bilan([comparaison], comparaison)


# --- 1. Verifier l'acceptation du multi-images, avant de s'en servir ---------
_parts_test = [prepare_image_for_gpt5(image_a), prepare_image_for_gpt5(image_b)]
# Budget par defaut, comme les appels de comparaison : GPT-5 depense d'abord
# des jetons de raisonnement, et un plafond serre (essaie a 16) laisse le canal
# texte vide -- HTTP 200, reponse vide, mesure le 2026-11-05.
_sonde = appel_vision(_parts_test, "Combien d'images distingues-tu ? Reponds par un chiffre.")
# L'acceptation se lit dans la REPONSE du modele, pas dans le seul HTTP 200 :
# un modele qui ignorerait la seconde image repondrait "1" avec le meme code 200.
_reponse_sonde = (_sonde.get("texte") or "").strip()
_MULTI_IMAGES_OK = _sonde["ok"] and any(mot in _reponse_sonde for mot in ("2", "deux", "two"))
print(f"Reponse de la sonde ({_sonde['parts_images']} images envoyees) : {_reponse_sonde!r}")
if _MULTI_IMAGES_OK:
    print(f"Multi-images accepte par {model_name} : la reponse cite bien deux images")
else:
    if not _sonde["ok"]:
        print(f"Multi-images REFUSE par {model_name} : {_sonde['erreur']}")
    else:
        print(f"Multi-images NON CONFIRME par {model_name} : HTTP 200, "
              "mais la reponse ne cite pas deux images")
    print("Les cellules suivantes ne peuvent pas s'executer : le modele ne lit qu'une image par message.")

# --- 2. La meme consigne de comparaison pour les deux voies ------------------
prompt_comparaison = f"""Tu compares les images jointes pour des étudiants. Réponds en {language}.
**Ce que montre chaque image** : une puce par image, avec un détail précis (compte les éléments quand c'est possible)
**Ce que la seconde ajoute à la première** : 2 puces
**Verdict** : laquelle introduit le domaine à des étudiants, en 2 lignes"""

if _MULTI_IMAGES_OK:
    verdict_descriptions, cout_descriptions = comparer_par_descriptions(
        image_a, image_b, prompt_analyse, prompt_comparaison)
    verdict_direct, cout_direct = comparer_directement(
        [image_a, image_b], prompt_comparaison)

    print("\nVoie 'descriptions' (exemple precedent) :", cout_descriptions)
    print("Voie directe (images jointes)            :", cout_direct)
    gain_calls = cout_descriptions["appels"] - cout_direct["appels"]
    print(f"\nAppels economises : {gain_calls}  |  "
          f"images vues a la comparaison : {cout_descriptions['images_vues_a_la_comparaison']} "
          f"-> {cout_direct['images_vues_a_la_comparaison']}")

    print("\n" + "=" * 30 + " VOIE DESCRIPTIONS " + "=" * 30)
    print(verdict_descriptions)
    print("\n" + "=" * 30 + " VOIE DIRECTE " + "=" * 30)
    print(verdict_direct)
else:
    verdict_descriptions, verdict_direct = None, None
    cout_descriptions, cout_direct = None, None
    print("Comparaison sautee : le modele ne lit qu'une image par message.")

Reponse de la sonde (2 images envoyees) : '2'
Multi-images accepte par openai/gpt-5 : la reponse cite bien deux images



Voie 'descriptions' (exemple precedent) : {'appels': 3, 'images_vues_a_la_comparaison': 1, 'tokens_entree': 1740, 'tokens_sortie': 4173, 'secondes': 54.5}
Voie directe (images jointes)            : {'appels': 1, 'images_vues_a_la_comparaison': 2, 'tokens_entree': 925, 'tokens_sortie': 1701, 'secondes': 22.3}

Appels economises : 2  |  images vues a la comparaison : 1 -> 2

============================== VOIE DESCRIPTIONS ==============================
Ce que montre chaque image
- Image A : Photo de haute montagne au crépuscule avec 1 mer de nuages remplissant la vallée, sommets enneigés et inversion thermique visible.
- Image B : Carte politique de la France métropolitaine + Corse montrant 22 régions (avant 2016) et 96 départements numérotés, 4 grands fleuves nommés et 3 massifs majeurs, le tout bordé par 3 façades maritimes.

Ce que la seconde ajoute à la première
- Des repères toponymiques et administratifs quantifiables (régions, départements), avec échelle et orientation pour mesu

### Lecture du résultat — ce que la comparaison directe change, et ce qu'elle ne change pas

L'acceptation du multi-images a été **mesurée** avant usage : interrogé sur le nombre d'images qu'il distingue dans un message portant deux parties `image_url`, `openai/gpt-5` répond `2`. Le reste de la cellule s'appuie sur cette réponse, pas sur la réputation du modèle.

Les deux voies, sur la même paire d'images et avec la même consigne de comparaison :

| | voie « descriptions » | voie directe |
|---|---|---|
| appels | 3 | **1** |
| images vues à la comparaison | 1 | **2** |
| tokens d'entrée | 1 740 | **925** |
| tokens de sortie | 4 173 | **1 701** |
| durée | 54,5 s | **22,3 s** |

Trois lectures, dans l'ordre de ce qui est solide :

1. **La mesure qui tranche est structurelle, pas qualitative.** Le croisement de la voie « descriptions » reçoit **une** image : tout ce qu'il affirme sur l'image A vient d'un texte, sans recours possible. La voie directe en reçoit deux. Aucune formulation de prompt ne corrige cette asymétrie — c'est le montage qui la crée.
2. **La voie texte paie deux fois.** Elle produit deux descriptions complètes (d'où les 4 173 tokens de sortie), puis les relit dans un troisième appel. La voie directe n'écrit qu'une réponse : moins d'appels, moins de tokens dans les deux sens, et ici 2,4 fois moins de temps.
3. **Ce que la voie directe voit et que le texte manque** : elle **compte** — « environ 10 sommets visibles » sur l'image A. Aucune des deux descriptions ne l'avait produit ; la voie texte ne pouvait donc pas le relayer. Un compte se lit sur les pixels, pas dans un résumé.

Et une lecture inverse, pour ne pas surinterpréter : **les deux verdicts choisissent la même image** (la carte, B). La comparaison directe ne promet pas un meilleur verdict — elle supprime un intermédiaire où l'information peut se perdre, et l'asymétrie entre les deux images. Ce qui reste fragile dans la voie texte est ailleurs : les nombres qu'elle relaie (22 anciennes régions, 96 départements) viennent d'une seule description, et rien dans la chaîne ne les contredit ; la voie directe les lit sur la carte.

Coût de la section : cinq appels de vision sur `openai/gpt-5` — la sonde d'acceptation, les trois appels de la voie texte, l'appel de la voie directe. Un modèle plus léger suffirait à ces appels, l'un des cinq en particulier : la comparaison directe n'en demande qu'un. Non mesuré ici.

## Exercices à compléter

Les trois exercices qui suivent portent sur ce que ces exemples ne mesurent pas encore : la **stabilité** d'une sortie répétée à prompt constant, la **fidélité** d'un alt text aux éléments réellement visibles, et la **structuration** d'une note d'évaluation en JSON validable. Chaque squelette s'exécute sans erreur tel quel — à vous de remplir les `TODO etudiant`.

### Exercice 1 : Reproductibilité — la même question posée trois fois

L'exemple guidé *prompt neutre vs prompt orienté* mesurait l'effet de l'**angle** du prompt. Ici, tout est figé : même image, même prompt, même modèle. La question devient donc *la sortie est-elle stable ?* Un modèle d'inférence échantillonne, et deux appels identiques peuvent ne pas rendre le même texte.

**Objectif** : mesurer la dispersion de trois analyses identiques, puis examiner ce que l'API **annonce** comme levier de stabilisation.

**Un fait à connaître avant l'étape 4** — vérifié sur l'API : `openai/gpt-5` ne liste **ni `temperature` ni `top_p`** dans ses paramètres supportés (champ `supported_parameters` de `GET /models/openai/gpt-5/endpoints` : `reasoning`, `seed`, `max_tokens`…). Cela établit une **absence de support annoncé**, pas un comportement : les métadonnées ne disent pas ce que le fournisseur fait d'une valeur `temperature` envoyée quand même (rejet, routage, normalisation) — ce comportement réel se vérifie dans la réponse API, pas dans les métadonnées. Le paramètre de stabilisation **annoncé** est `seed`.

**Étapes** :
1. Choisissez une image et un prompt court à sections fixes
2. Lancez **trois** analyses identiques et conservez les trois textes
3. Mesurez la similarité deux à deux — un Jaccard lexical comme dans l'exemple guidé (réutilisez `mots_cles`), ou un simple taux de sections communes — et présentez le résultat comme **descriptif**
4. Consultez les `supported_parameters` du modèle et concluez sur le support **annoncé** de `temperature`, sans en inférer un comportement
5. Concluez en une phrase : quel levier l'API annonce-t-elle pour rapprocher deux exécutions, et pour quel usage faut-il une sortie stable ?

**Facultatif — deux appels payants sur un paramètre non annoncé** : le squelette propose en étape 4 de réassigner `temperature = 0.0` puis de relancer. L'expérience est faisable mais **non concluante en l'état** : comparer la dispersion de trois échantillons à celle de deux échantillons ne prouve ni honoration ni non-honoration du paramètre — c'est une observation descriptive, à présenter comme telle. Si vous la faites, remettez `temperature = 0.7` en fin de cellule.

**Extension — le levier annoncé** : `seed` n'est pas exposé par `analyze_image_with_gpt5()`. L'utiliser suppose d'**adapter explicitement la fonction** (ajouter `"seed": ...` au payload de la requête), puis de mesurer si deux exécutions à `seed` fixé se rapprochent effectivement.

**Indices** :
- `analyze_image_with_gpt5()` lit `temperature` dans les variables globales du carnet : une réaffectation en cellule suffit, sans modifier la fonction
- paramètres supportés d'un modèle : `GET https://openrouter.ai/api/v1/models/openai/gpt-5/endpoints`, champ `supported_parameters`
- trois appels identiques coûtent trois fois le prix d'un appel — gardez un prompt court (`analysis_type="quick"` ou peu de sections)
- un Jaccard proche de 1 signale deux sorties quasi identiques, un Jaccard bas une reformulation complète

In [19]:
# Exercice 1 : Reproductibilite -- la meme question posee trois fois
# TODO etudiant : mesurer la dispersion de trois analyses identiques, puis a temperature = 0.0

# Etape 1 : Image et prompt fixes (aucune variation d'angle ici)
image_exo1_rep = example_images[3]['url']
prompt_repete = f"""Réponds en {language}, avec exactement ces 2 sections :
**1. Éléments visibles** : 3 puces
**2. Conclusion** : une seule phrase"""

# Etape 2 : Trois analyses identiques
# TODO etudiant : construire une liste de trois analyses via
#                 analyze_image_with_gpt5(image_exo1_rep, prompt=prompt_repete)
analyses_repetees = []  # TODO etudiant

# Etape 3 : Similarite deux a deux
# TODO etudiant : calculer la similarite de chaque paire (Jaccard lexical via mots_cles,
#                 ou taux de sections communes) et afficher les valeurs
similarites = []  # TODO etudiant

# Etape 4 : Effet de la temperature
# TODO etudiant : reassigner temperature = 0.0, relancer deux analyses, comparer la dispersion,
#                 puis restaurer temperature = 0.7
analyses_froides = []  # TODO etudiant

# Etape 5 : Conclusion en une phrase (a afficher)
print("Exercice a completer")

Exercice a completer


### Exercice 2 : Audit de fidélité — un alt text n'affirme que ce qui est visible

L'exemple guidé *alt text contextuel* produisait trois textes conformes à une limite de longueur et à un contexte. Une qualité échappe à ces deux contrôles : la **fidélité**. Un alt text court et bien tourné peut citer une couleur, un objet ou un lieu que l'image ne contient pas — et le lecteur d'écran ne peut pas le contredire.

**Objectif** : construire un audit en deux temps — un inventaire des éléments visibles, puis un alt text vérifié terme à terme contre cet inventaire.

**Étapes** :
1. Choisissez une image parmi `example_images`
2. Demandez un **inventaire** : liste à puces des éléments visibles, sans interprétation
3. Demandez un **alt text** de 125 caractères maximum pour la même image
4. Extrayez les termes significatifs de l'alt text (réutilisez `mots_cles`) et relevez ceux qui n'apparaissent pas dans l'inventaire
5. Affichez le verdict et rappelez la limite de la méthode

**Pourquoi deux appels séparés ?** Les appels à `analyze_image_with_gpt5()` sont **indépendants** : aucun historique conversationnel ne circule de l'un à l'autre, et l'ordre des deux demandes n'a donc pas d'effet causal. L'intérêt de la séparation est ailleurs : deux prompts courts et ciblés, au lieu d'un prompt fusionné où le modèle verrait l'alt text au moment de dresser l'inventaire — et répondrait en conséquence à sa propre conclusion.

**Indices** :
- un terme absent de l'inventaire n'est pas nécessairement inventé (synonyme, reformulation) : le verdict dit « à réviser », pas « faux »
- l'inventaire et l'alt text sont deux **mesures distinctes** de la même image : les comparer n'a de sens que si chacune a été produite sous une consigne unique

In [20]:
# Exercice 2 : Audit de fidelite -- un alt text n'affirme que ce qui est visible
# TODO etudiant : inventaire des elements visibles, alt text, puis verification terme a terme

# Etape 1 : Image
image_exo2_audit = example_images[2]['url']

# Etape 2 : Inventaire des elements visibles (demande AVANT l'alt text)
prompt_inventaire = f"""Liste en {language} les éléments visibles de cette image.
Réponds uniquement par une liste à puces de 5 à 8 éléments factuels, sans interprétation."""
# TODO etudiant : inventaire = analyze_image_with_gpt5(image_exo2_audit, prompt=prompt_inventaire)
inventaire = None  # TODO etudiant

# Etape 3 : Alt text soumis a l'audit
prompt_alt_audit = f"""Génère un alt text en {language} de 125 caractères maximum,
une seule phrase factuelle, sans interprétation subjective."""
# TODO etudiant : alt_text_audit = analyze_image_with_gpt5(image_exo2_audit, prompt=prompt_alt_audit)
alt_text_audit = None  # TODO etudiant

# Etape 4 : Verification terme a terme contre l'inventaire
# TODO etudiant : termes de l'alt text (mots_cles) absents de l'inventaire (mots_cles)
termes_non_etayes = []  # TODO etudiant

# Etape 5 : Verdict
# TODO etudiant : afficher la liste des termes non etayes et rappeler la limite de la
#                 verification lexicale (un synonyme n'est pas une invention)
print("Exercice a completer")

Exercice a completer


### Exercice 3 : Évaluation structurée — noter une réponse en JSON validable

Les exemples guidés produisent des analyses en texte libre. Un usage courant en enseignement est l'inverse : **noter** une réponse selon des critères explicites, et rendre cette note exploitable par un programme. La difficulté n'est pas la note elle-même, c'est d'obtenir une réponse **structurée et validable** — un modèle qui rend une phrase d'appréciation ne rend pas une note.

**Objectif** : écrire une fonction qui évalue un alt text selon trois critères et retourne un JSON strict, puis **valider** ce JSON (parsing, clés, bornes).

**Étapes** :
1. Choisissez l'image et la réponse à noter (un alt text généré plus haut convient)
2. Définissez une rubrique de trois critères, chacun noté de 0 à 2
3. Rédigez un prompt qui exige **uniquement** un objet JSON avec les clés attendues — sans préambule ni bloc de code markdown
4. Parsez la réponse (`json.loads`) et validez : trois critères présents, chaque note entière et comprise entre 0 et 2
5. Affichez la note totale, et **signalez** les cas où le parsing échoue au lieu de les contourner

**Indices** :
- `json.loads()` échoue sur un bloc de code markdown : demandez le texte brut, sans balises
- une validation stricte (`isinstance(..., int)` et bornes) transforme « ça a l'air bon » en verdict binaire
- l'évaluateur est un modèle **stochastique** : la note est indicative, pas une vérité — sa propre répétabilité se mesure (relancez l'évaluation à l'identique et comparez les deux notes)


In [21]:
# Exercice 3 : Evaluation structuree -- noter un alt text avec une rubrique, en JSON
# TODO etudiant : obtenir une note JSON valide (3 criteres, chacun de 0 a 2)

# Etape 1 : La reponse a evaluer
reponse_a_noter = "Vue aérienne d'une ville au crépuscule avec un dôme doré et des montagnes."  # TODO etudiant : votre propre alt text
image_eval = example_images[0]['url']

# Etape 2 : La rubrique (3 criteres, notes de 0 a 2)
RUBRIQUE = {
    "fidelite": "0-2 : chaque element de l'alt text est visible sur l'image",
    "longueur": "0-2 : respecte la limite de 125 caracteres",
    "objectivite": "0-2 : aucune interpretation subjective",
}

# Etape 3 : Prompt exigeant un JSON strict
# TODO etudiant : rediger le prompt (rubrique + reponse + format JSON exact, sans preambule)

# Etape 4 : Appel, parsing et validation
# TODO etudiant : note = json.loads(reponse) puis verifier les cles, le type entier et les bornes
note_json = None  # TODO etudiant

# Etape 5 : Affichage du verdict
print("Exercice a completer")

Exercice a completer


### Critères de succès

**Exercice 1 — reproductibilité**
- [ ] Trois analyses identiques lancées, et leurs similarités deux à deux mesurées (pas estimées), présentées comme **descriptives**
- [ ] Les `supported_parameters` du modèle consultés, la conclusion limitée au support **annoncé** de `temperature` — aucun comportement (ignoration, rejet, normalisation) inféré des seules métadonnées
- [ ] Toute comparaison d'échantillons additionnels (facultative, payante) reste descriptive, jamais présentée comme preuve causale ; `temperature` est remise à `0.7` en fin de cellule si l'expérience est faite

**Exercice 2 — audit de fidélité**
- [ ] Inventaire et alt text produits par deux appels **séparés**, chacun sous une consigne unique
- [ ] Les termes de l'alt text sont confrontés à l'inventaire par un calcul, avec la liste des termes non étayés
- [ ] Le verdict dit « à réviser » et nomme la limite de la vérification lexicale

**Exercice 3 — évaluation structurée**
- [ ] La rubrique compte trois critères notés de 0 à 2
- [ ] Le JSON est parsé et validé (clés présentes, entiers, bornes) — un échec de parsing est signalé, jamais contourné
- [ ] La note totale et les cas d'échec sont affichés, avec la nature **indicative** de la note rappelée

### Extension (optionnelle)

Pour les étudiants avancés :
- Exposer `seed` dans `analyze_image_with_gpt5()` (adaptation explicite du payload) et mesurer si deux exécutions à `seed` fixé se rapprochent
- Créer un **quiz interactif** basé sur les questions générées
- Étendre le comparateur à **trois images** ou plus (matrice de comparaison)
- Rejouer l'exercice 3 avec une seconde évaluation indépendante (même rubrique, prompt différent) et mesurer l'accord entre les deux notes

### Lecture du résultat — les exercices en synthèse

Les trois exemples guidés de la section précédente montrent chacun un geste mesuré : comparer deux formulations à format constant, adapter un alt text à son contexte sous contrainte de longueur, croiser deux analyses avec un prompt identique. Les trois exercices qui les complètent portent sur ce que ces gestes ne mesurent pas encore : la **stabilité** d'une sortie répétée, la **fidélité** d'un alt text aux éléments visibles, et la **structuration** d'une note en JSON validable. Le carnet s'exécute de bout en bout tant que les exercices restent des stubs.

## Conclusion

GPT-5 image generation (via OpenRouter) incarne l'approche **multimodale native** : le modèle ne se contente pas de générer des images à partir d'un prompt texte, il analyse et produit dans un même flux, ce qui ouvre trois modes d'usage pédagogique distincts — quick, detailed, educational — richesse et coût croissent ensemble (les mesures vivent dans la table imprimée par le test). Le ratio coût d'inférence / richesse du contenu est directement exposé à l'utilisateur par le choix du mode.

**Apport spécifique de la multimodalité native** : la génération automatique d'alt text (une phrase unique sous le seuil de 125 caractères imposé par le prompt) intègre la description d'image comme **livrable de première classe**, pas comme un post-traitement optionnel. Pour les 5 cas d'usage éducatif couverts (Histoire, Sciences, Géographie, Art et Culture, Médecine), c'est un gain d'accessibilité substantiel par rapport aux modèles purement text-to-image.

**Positionnement dans l'écosystème** : GPT-5 complète la palette du cours en couvrant l'axe *compréhension + génération*, là où DALL-E 3 se concentre sur la génération seule et où SD XL Turbo expose les rouages internes. La large famille de modèles GPT-5 exposée par l'API OpenRouter ouvrent un terrain d'expérimentation comparative large mais introduisent une dépendance forte au fournisseur de route.

**Pour aller plus loin** : explorer le notebook `04-1-Educational-Content-Generation` qui orchestre plusieurs modèles pour produire un livrable pédagogique complet (image + alt text + métadonnées structurées).